# Dihedral Distributions & Conformational Clustering — MUC2 Long PTS (3 Systems)

**Three glycosylation states compared:**
- **Non-glycosylated** — naked PTS backbone
- **Mid-level Glycosylated** — partial O-glycosylation
- **Ext. Glycosylated** — full O-glycosylation (parsed from 3 representative PDB files)

**Analyses:**
1. 2-D KDE Ramachandran plot per system
2. 1-D phi and psi distributions (KDE + histogram overlay)
3. K-means conformational clustering (k=4 basins)
4. Cluster population comparison across systems
5. Pre-glycan residue cluster membership

In [ ]:
import os, csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
from scipy.stats import gaussian_kde
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

BASE    = '/home/computer2/analysis_simulations/analysis/dihedral_angles_15structures_mean_structure/'
PDB_DIR = '/home/computer2/analysis_simulations/analysis/pdb_structures/'
output_dir = '/home/computer2/analysis_simulations/analysis/dihedral_clustering_plots'
os.makedirs(output_dir, exist_ok=True)

RES_OFFSET = 1377
CLR_NOG = '#A23B72'   # non-glycosylated  — purple/magenta
CLR_GLY = '#2E86AB'   # mid-level glyc    — blue
CLR_EXT = '#F18F01'   # ext. glycosylated — orange

GLYCAN_RES = np.array([1397,1400,1403,1406,1409,1412,
                        1420,1423,1426,1429,1432,1435,
                        1443,1446,1449,1452,1455,1458,
                        1466,1469,1472,1475,1478,1481,
                        1489,1492,1495,1498,1501,1504])
PRE_RES = GLYCAN_RES - 1

print('Ready.  Output:', output_dir)

## 1 — Load Data

In [ ]:
# ── Load pre-computed distributions (noglycans & glycans) ────────────────
def load_dat(filepath):
    phi, psi, res, frame = [], [], [], []
    with open(filepath) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            phi.append(float(row['phi_deg']))
            psi.append(float(row['psi_deg']))
            res.append(int(row['residue_index']))
            frame.append(int(row['frame_rank']))
    return (np.array(phi), np.array(psi),
            np.array(res) + RES_OFFSET, np.array(frame))

phi_nog, psi_nog, res_nog, fr_nog = load_dat(BASE + 'dihedral_distribution_noglycans.dat')
phi_gly, psi_gly, res_gly, fr_gly = load_dat(BASE + 'dihedral_distribution_glycans.dat')

# ── Parse phi/psi for ext. glycosylated from 3 PDB files ─────────────────
PROTEIN_RES = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU',
    'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL','HID','HIE','HIP','CYX'
}

def dihedral(p0, p1, p2, p3):
    b0 = -1.0*(p1-p0); b1 = p2-p1; b2 = p3-p2
    b1 /= np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1)*b1
    w = b2 - np.dot(b2, b1)*b1
    return np.degrees(np.arctan2(np.dot(np.cross(b1, v), w), np.dot(v, w)))

def parse_phi_psi(pdb_path):
    N, CA, C = {}, {}, {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith('ATOM'): continue
            name = line[12:16].strip(); rn = line[17:20].strip()
            if rn not in PROTEIN_RES: continue
            resn = int(line[22:26])
            xyz  = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            if name == 'N':  N[resn]  = xyz
            elif name == 'CA': CA[resn] = xyz
            elif name == 'C':  C[resn]  = xyz
    residues = sorted(set(N) & set(CA) & set(C))
    phi, psi, ri = [], [], []
    for i, r in enumerate(residues):
        if i == 0 or i == len(residues)-1: continue
        prev, nxt = residues[i-1], residues[i+1]
        if prev not in C or nxt not in N: continue
        phi.append(dihedral(C[prev], N[r], CA[r], C[r]))
        psi.append(dihedral(N[r], CA[r], C[r], N[nxt]))
        ri.append(r)
    return np.array(phi), np.array(psi), np.array(ri)

phi_ext_all, psi_ext_all, res_ext_all = [], [], []
for s in [1, 2, 3]:
    phi, psi, ri = parse_phi_psi(f'{PDB_DIR}ext_glycans_struct{s}.pdb')
    phi_ext_all.append(phi); psi_ext_all.append(psi); res_ext_all.append(ri)
phi_ext = np.concatenate(phi_ext_all)
psi_ext = np.concatenate(psi_ext_all)
res_ext = np.concatenate(res_ext_all)

print(f'Non-glyc     : {len(phi_nog):5d} phi/psi pairs  ({len(np.unique(fr_nog))} structures × {len(np.unique(res_nog))} residues)')
print(f'Mid-glyc     : {len(phi_gly):5d} phi/psi pairs')
print(f'Ext. glyc    : {len(phi_ext):5d} phi/psi pairs  (3 representative PDB files)')


## 2 — 2-D KDE Ramachandran Plot

In [ ]:
systems = [
    ('Non-glycosylated',  phi_nog, psi_nog, CLR_NOG),
    ('Mid-level Glyc',   phi_gly, psi_gly, CLR_GLY),
    ('Ext. Glycosylated', phi_ext, psi_ext, CLR_EXT),
]

fig, axes = plt.subplots(1, 3, figsize=(19, 6), dpi=130)
fig.set_facecolor('white')

bins = np.linspace(-180, 180, 73)
vmin, vmax = 1, None  # log scale

for ax, (name, phi, psi, clr) in zip(axes, systems):
    h, xe, ye = np.histogram2d(phi, psi, bins=bins)
    h = np.ma.masked_where(h == 0, h)
    im = ax.pcolormesh(xe, ye, h.T, cmap='hot_r', norm=LogNorm(vmin=1))
    ax.set_xlabel('φ (degrees)', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
    ax.set_ylabel('ψ (degrees)', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
    ax.set_title(name, fontsize=14, fontfamily='DejaVu Serif', fontweight='bold', color=clr)
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
    ax.axhline(0, color='grey', lw=0.6, ls='--', alpha=0.5)
    ax.axvline(0, color='grey', lw=0.6, ls='--', alpha=0.5)
    ax.set_xticks([-120, -60, 0, 60, 120])
    ax.set_yticks([-120, -60, 0, 60, 120])
    ax.tick_params(labelsize=10)
    plt.colorbar(im, ax=ax, label='Count (log scale)')
    # annotate conformational regions
    for (rx, ry, rlabel) in [(-75, 145, 'PPII'), (-120, 130, 'β'), (-60, -40, 'αR')]:
        ax.text(rx, ry, rlabel, fontsize=10, color='navy',
                fontfamily='DejaVu Serif', fontweight='bold', alpha=0.8,
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='navy', alpha=0.7))
    for sp in ax.spines.values(): sp.set_linewidth(1.5)

fig.suptitle('Ramachandran Plot — MUC2 Long PTS (3 Systems)\n'
             '(PPII = polyproline II, β = extended strand, αR = right-handed helix)',
             fontsize=15, fontfamily='DejaVu Serif', fontweight='bold', y=1.01)
plt.tight_layout()
out = os.path.join(output_dir, 'ramachandran_3sys_kde.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

In [ ]:
# Overlay: all three systems on one Ramachandran
fig, ax = plt.subplots(figsize=(8, 7.5), dpi=130)
fig.set_facecolor('white')

for name, phi, psi, clr in systems:
    h, xe, ye = np.histogram2d(phi, psi, bins=np.linspace(-180,180,61))
    h = h / h.sum()  # normalise to density
    lvls = np.percentile(h[h>0], [50, 70, 85, 95])
    ax.contour(0.5*(xe[:-1]+xe[1:]), 0.5*(ye[:-1]+ye[1:]), h.T,
               levels=lvls, colors=[clr], linewidths=[0.6,0.9,1.2,1.8],
               alpha=0.85)
    ax.contourf(0.5*(xe[:-1]+xe[1:]), 0.5*(ye[:-1]+ye[1:]), h.T,
                levels=[lvls[-1], h.max()], colors=[clr], alpha=0.18)

# Fake legend
patches = [mpatches.Patch(color=clr, alpha=0.7, label=name)
           for name, _, _, clr in systems]
ax.legend(handles=patches, fontsize=11, prop={'family':'DejaVu Serif','size':11},
          loc='lower right')
ax.set_xlabel('φ (degrees)', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
ax.set_ylabel('ψ (degrees)', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
ax.set_xlim(-180,180); ax.set_ylim(-180,180)
ax.axhline(0, color='grey', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='grey', lw=0.6, ls='--', alpha=0.5)
for (rx, ry, rlabel) in [(-75,145,'PPII'),(-120,130,'β'),(-60,-40,'αR')]:
    ax.text(rx, ry, rlabel, fontsize=12, color='#1A252F',
            fontfamily='DejaVu Serif', fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.25', fc='#F8F9FA', ec='#1A252F', alpha=0.9))
ax.set_title('Ramachandran Overlay — 3 Systems\n(contours at 50/70/85/95th density percentile)',
             fontsize=14, fontfamily='DejaVu Serif', fontweight='bold')
ax.tick_params(labelsize=10)
for sp in ax.spines.values(): sp.set_linewidth(1.5)
plt.tight_layout()
out = os.path.join(output_dir, 'ramachandran_overlay.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

## 3 — 1-D Phi & Psi Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10), dpi=130)
fig.set_facecolor('white')
bins1d = np.linspace(-180, 180, 73)

for col, (name, phi, psi, clr) in enumerate(systems):
    for row, (angles, label) in enumerate([(phi, 'φ (degrees)'), (psi, 'ψ (degrees)')]):
        ax = axes[row, col]
        ax.hist(angles, bins=bins1d, color=clr, alpha=0.45,
                density=True, edgecolor='none', label='histogram')
        # KDE
        kde = gaussian_kde(angles, bw_method=0.12)
        x_k = np.linspace(-180, 180, 400)
        ax.plot(x_k, kde(x_k), color=clr, lw=2.2, label='KDE')
        ax.set_xlabel(label, fontsize=11, fontfamily='DejaVu Serif', fontweight='bold')
        ax.set_ylabel('Density', fontsize=11, fontfamily='DejaVu Serif', fontweight='bold')
        ax.set_xlim(-180, 180)
        ax.set_title(f'{name}\n{label.split()[0]} distribution',
                     fontsize=11, fontfamily='DejaVu Serif', fontweight='bold', color=clr)
        ax.tick_params(labelsize=9)
        ax.axvline(np.median(angles), color='black', lw=1.2, ls='--', alpha=0.7,
                   label=f'median={np.median(angles):.0f}°')
        ax.legend(fontsize=9, prop={'family':'DejaVu Serif','size':9})
        for sp in ax.spines.values(): sp.set_linewidth(1.4)

fig.suptitle('Phi (φ) and Psi (ψ) Distributions — MUC2 Long PTS (3 Systems)',
             fontsize=15, fontfamily='DejaVu Serif', fontweight='bold', y=1.01)
plt.tight_layout()
out = os.path.join(output_dir, 'phi_psi_distributions_1d.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

In [ ]:
fig, (ax_phi, ax_psi) = plt.subplots(1, 2, figsize=(16, 6), dpi=130)
fig.set_facecolor('white')
x_k = np.linspace(-180, 180, 500)

for name, phi, psi, clr in systems:
    kde_phi = gaussian_kde(phi, bw_method=0.10)
    kde_psi = gaussian_kde(psi, bw_method=0.10)
    ax_phi.plot(x_k, kde_phi(x_k), color=clr, lw=2.5, label=name)
    ax_phi.fill_between(x_k, kde_phi(x_k), color=clr, alpha=0.12)
    ax_psi.plot(x_k, kde_psi(x_k), color=clr, lw=2.5, label=name)
    ax_psi.fill_between(x_k, kde_psi(x_k), color=clr, alpha=0.12)

for ax, label, regions in [
    (ax_phi, 'φ (degrees)',
     [(-75, 'PPII\nφ≈-75°'), (-120, 'β-strand\nφ≈-120°'), (-60, 'αR\nφ≈-60°')]),
    (ax_psi, 'ψ (degrees)',
     [(145, 'PPII\nψ≈+145°'), (130, 'β-strand\nψ≈+130°'), (-40, 'αR\nψ≈-40°')]),
]:
    ax.set_xlabel(label, fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
    ax.set_ylabel('Density', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
    ax.set_xlim(-180, 180)
    ax.set_title(f'{label.split()[0]} Distribution Overlay',
                 fontsize=14, fontfamily='DejaVu Serif', fontweight='bold')
    ax.legend(fontsize=11, prop={'family':'DejaVu Serif','size':11})
    ax.tick_params(labelsize=10)
    for rv, rlabel in regions:
        ax.axvline(rv, color='grey', lw=1.0, ls=':', alpha=0.7)
        ax.text(rv, ax.get_ylim()[1]*0.95 if ax.get_ylim()[1]>0 else 0.05,
                rlabel, ha='center', va='top', fontsize=8,
                fontfamily='DejaVu Serif', color='#444',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='grey', alpha=0.8))
    for sp in ax.spines.values(): sp.set_linewidth(1.5)

plt.tight_layout()
out = os.path.join(output_dir, 'phi_psi_kde_overlay.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

## 4 — K-means Conformational Clustering

K-means in (φ, ψ) space with **k = 4** known conformational basins:
- **PPII** (polyproline II): φ ≈ -75°, ψ ≈ +145°
- **β-strand**: φ ≈ -120°, ψ ≈ +130°
- **αR-helix**: φ ≈ -60°, ψ ≈ -40°
- **Other** (αL, turns, etc.): φ ≈ +60°, ψ ≈ +40°

In [ ]:
# ── Fit k-means on pooled data ────────────────────────────────────────────
phi_all = np.concatenate([phi_nog, phi_gly, phi_ext])
psi_all = np.concatenate([psi_nog, psi_gly, psi_ext])
X_all   = np.column_stack([phi_all, psi_all])

# Provide good starting centroids for the 4 known basins
init_centroids = np.array([
    [-75,  145],   # PPII
    [-120, 130],   # β-strand
    [-60,  -40],   # αR-helix
    [ 60,   40],   # other (αL / turn)
])

km = KMeans(n_clusters=4, init=init_centroids, n_init=1, random_state=42)
km.fit(X_all)

# Assign meaningful labels to cluster IDs based on centroid location
centroid_phi = km.cluster_centers_[:, 0]
# Sort: most-negative phi → β; second → PPII; third → αR; positive → other
order = np.argsort(centroid_phi)  # ascending phi

CLUSTER_NAMES = {}
CLUSTER_COLORS = {}
name_seq  = ['β-strand', 'PPII', 'αR-helix', 'Other']
color_seq = ['#2ECC71', '#3498DB', '#E74C3C', '#95A5A6']

for rank, cid in enumerate(order):
    CLUSTER_NAMES[cid]  = name_seq[rank]
    CLUSTER_COLORS[cid] = color_seq[rank]

print('Cluster centroids:')
for cid in range(4):
    phi_c, psi_c = km.cluster_centers_[cid]
    print(f'  Cluster {cid}  ({CLUSTER_NAMES[cid]:10s}): φ={phi_c:+.1f}°  ψ={psi_c:+.1f}°')

In [ ]:
# ── Assign cluster labels per system ─────────────────────────────────────
def cluster_counts(phi, psi):
    labels = km.predict(np.column_stack([phi, psi]))
    counts = {CLUSTER_NAMES[c]: np.sum(labels == c) for c in range(4)}
    total  = len(labels)
    fracs  = {k: v/total*100 for k, v in counts.items()}
    return labels, fracs

labels_nog, fracs_nog = cluster_counts(phi_nog, psi_nog)
labels_gly, fracs_gly = cluster_counts(phi_gly, psi_gly)
labels_ext, fracs_ext = cluster_counts(phi_ext, psi_ext)

print('Cluster populations (%):')
print(f'{"Basin":12s}  {"NoGlyc":>9s}  {"MidGlyc":>9s}  {"ExtGlyc":>9s}')
for basin in ['PPII', 'β-strand', 'αR-helix', 'Other']:
    print(f'{basin:12s}  {fracs_nog.get(basin,0):9.1f}  {fracs_gly.get(basin,0):9.1f}  {fracs_ext.get(basin,0):9.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5), dpi=130)
fig.set_facecolor('white')

system_data = [
    ('Non-glycosylated', phi_nog, psi_nog, labels_nog),
    ('Mid-level Glyc',   phi_gly, psi_gly, labels_gly),
    ('Ext. Glycosylated', phi_ext, psi_ext, labels_ext),
]

for ax, (name, phi, psi, labels) in zip(axes, system_data):
    for cid in range(4):
        mask = labels == cid
        ax.scatter(phi[mask], psi[mask], s=12, alpha=0.55,
                   color=CLUSTER_COLORS[cid], label=CLUSTER_NAMES[cid], rasterized=True)
    # plot centroids
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               s=160, marker='*', c='black', zorder=5, label='Centroids')
    ax.set_xlabel('φ (°)', fontsize=12, fontfamily='DejaVu Serif', fontweight='bold')
    ax.set_ylabel('ψ (°)', fontsize=12, fontfamily='DejaVu Serif', fontweight='bold')
    ax.set_xlim(-180,180); ax.set_ylim(-180,180)
    ax.axhline(0, color='grey', lw=0.5, ls='--', alpha=0.5)
    ax.axvline(0, color='grey', lw=0.5, ls='--', alpha=0.5)
    ax.set_title(name, fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
    ax.legend(fontsize=9, prop={'family':'DejaVu Serif','size':9}, markerscale=2)
    ax.tick_params(labelsize=9)
    for sp in ax.spines.values(): sp.set_linewidth(1.5)

fig.suptitle('K-means Conformational Clusters (k=4) — MUC2 Long PTS',
             fontsize=15, fontfamily='DejaVu Serif', fontweight='bold', y=1.01)
plt.tight_layout()
out = os.path.join(output_dir, 'kmeans_cluster_scatter.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

## 5 — Cluster Population Comparison

In [ ]:
basins = ['PPII', 'β-strand', 'αR-helix', 'Other']
sys_labels = ['Non-glycosylated', 'Mid-level Glyc', 'Ext. Glycosylated']
fracs_list = [fracs_nog, fracs_gly, fracs_ext]
sys_colors = [CLR_NOG, CLR_GLY, CLR_EXT]

x = np.arange(len(basins))
w = 0.26

fig, ax = plt.subplots(figsize=(13, 7), dpi=130)
fig.set_facecolor('white')

for i, (label, fracs, clr) in enumerate(zip(sys_labels, fracs_list, sys_colors)):
    vals = [fracs.get(b, 0) for b in basins]
    bars = ax.bar(x + (i-1)*w, vals, width=w, color=clr, alpha=0.88,
                  edgecolor='black', linewidth=0.5, label=label)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{val:.1f}%', ha='center', va='bottom',
                fontsize=9.5, fontfamily='DejaVu Serif', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='grey', alpha=0.85))

ax.set_xticks(x)
ax.set_xticklabels(basins, fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
ax.set_ylabel('Population (%)', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
ax.set_title('Conformational Basin Populations — 3 Glycosylation States\n'
             '(k-means clustering in φ/ψ space, k=4)',
             fontsize=14, fontfamily='DejaVu Serif', fontweight='bold')
ax.legend(fontsize=11, prop={'family':'DejaVu Serif','size':11})
ax.set_ylim(0, max([max(f.values()) for f in fracs_list]) + 10)
ax.tick_params(labelsize=11)
for sp in ax.spines.values(): sp.set_linewidth(1.6)

plt.tight_layout()
out = os.path.join(output_dir, 'cluster_population_bargraph.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

In [ ]:
# Stacked bar chart
fig, ax = plt.subplots(figsize=(9, 7), dpi=130)
fig.set_facecolor('white')

basin_colors = {b: c for b, c in zip(['PPII','β-strand','αR-helix','Other'],
                                       ['#3498DB','#2ECC71','#E74C3C','#95A5A6'])}
bottoms = np.zeros(3)
for basin in basins:
    vals = [fracs.get(basin, 0) for fracs in fracs_list]
    ax.bar(range(3), vals, bottom=bottoms, color=basin_colors[basin],
           edgecolor='white', linewidth=1.2, label=basin, alpha=0.92)
    for xi, (v, bot) in enumerate(zip(vals, bottoms)):
        if v > 3:
            ax.text(xi, bot + v/2, f'{v:.1f}%', ha='center', va='center',
                    fontsize=11, fontfamily='DejaVu Serif', fontweight='bold',
                    color='white' if basin != 'Other' else '#1A252F')
    bottoms += np.array(vals)

ax.set_xticks(range(3))
ax.set_xticklabels(sys_labels, fontsize=12, fontfamily='DejaVu Serif', fontweight='bold')
ax.set_ylabel('Population (%)', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
ax.set_title('Stacked Conformational Basin Populations',
             fontsize=14, fontfamily='DejaVu Serif', fontweight='bold')
ax.legend(fontsize=11, prop={'family':'DejaVu Serif','size':11},
          loc='lower right', bbox_to_anchor=(1.32, 0))
ax.set_ylim(0, 105)
ax.tick_params(labelsize=11)
for sp in ax.spines.values(): sp.set_linewidth(1.6)

plt.tight_layout()
out = os.path.join(output_dir, 'cluster_population_stacked.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

## 6 — Cluster Membership at Pre-glycan Residues

In [ ]:
# For each pre-glycan residue, compute cluster membership per system
THREE_TO_ONE = {'THR':'T','ILE':'I','VAL':'V','PRO':'P','GLY':'G','SER':'S','ALA':'A'}

seq_aa = {}
with open('/home/computer2/analysis_simulations/analysis/dihedral_angles_15structures_mean_structure/noglycans_mean.pdb') as fh:
    for line in fh:
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            seq_aa[int(line[22:26])] = line[17:20].strip()

pre_res_filtered = PRE_RES[(PRE_RES >= res_nog.min()) & (PRE_RES <= res_nog.max())]

# For each pre-glycan residue, collect all phi/psi from all frames
def get_preglycan_phi_psi(phi_arr, psi_arr, res_arr, target_res):
    mask = res_arr == target_res
    return phi_arr[mask], psi_arr[mask]

# Build per-residue cluster table for nog and gly
print(f'{'Res':>5} {'AA':>5}  {'NoGlyc_PPII%':>13} {'Gly_PPII%':>12} {'Ext_PPII%':>11}')
for pr in pre_res_filtered[:15]:  # show first 15
    aa3 = seq_aa.get(pr, '???')
    phi_n, psi_n = get_preglycan_phi_psi(phi_nog, psi_nog, res_nog, pr)
    phi_g, psi_g = get_preglycan_phi_psi(phi_gly, psi_gly, res_gly, pr)
    phi_e, psi_e = get_preglycan_phi_psi(phi_ext, psi_ext, res_ext, pr)
    def ppii_frac(ph, ps):
        if len(ph)==0: return float('nan')
        return np.sum((ph>-100)&(ph<-40)&(ps>100)&(ps<180)) / len(ph) * 100
    print(f'{pr:>5} {aa3:>5}  {ppii_frac(phi_n,psi_n):>13.1f} {ppii_frac(phi_g,psi_g):>12.1f} {ppii_frac(phi_e,psi_e):>11.1f}')

In [ ]:
# Plot PPII fraction at each pre-glycan residue across systems
fig, ax = plt.subplots(figsize=(20, 7), dpi=130)
fig.set_facecolor('white')

ppii_data = {clr: [] for clr in [CLR_NOG, CLR_GLY, CLR_EXT]}
labels_pr  = []

for pr in pre_res_filtered:
    aa3 = seq_aa.get(pr, '???')
    labels_pr.append(f'{pr}\n{aa3}')
    for phi_arr, psi_arr, res_arr, clr in [
        (phi_nog, psi_nog, res_nog, CLR_NOG),
        (phi_gly, psi_gly, res_gly, CLR_GLY),
        (phi_ext, psi_ext, res_ext, CLR_EXT),
    ]:
        ph, ps = get_preglycan_phi_psi(phi_arr, psi_arr, res_arr, pr)
        if len(ph) == 0:
            ppii_data[clr].append(np.nan)
        else:
            ppii_data[clr].append(np.sum((ph>-100)&(ph<-40)&(ps>100)&(ps<180))/len(ph)*100)

x = np.arange(len(pre_res_filtered))
w = 0.28
ax.bar(x - w, ppii_data[CLR_NOG], width=w, color=CLR_NOG, alpha=0.88,
       edgecolor='black', linewidth=0.4, label='Non-glycosylated')
ax.bar(x,     ppii_data[CLR_GLY], width=w, color=CLR_GLY, alpha=0.88,
       edgecolor='black', linewidth=0.4, label='Mid-level Glyc')
ax.bar(x + w, ppii_data[CLR_EXT], width=w, color=CLR_EXT, alpha=0.88,
       edgecolor='black', linewidth=0.4, label='Ext. Glycosylated')

for i in range(len(pre_res_filtered)):
    if i % 6 == 5:
        ax.axvspan(i - 0.5, i + 0.5, color='#F39C12', alpha=0.12, zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(labels_pr, fontsize=8, fontfamily='DejaVu Serif', rotation=60, ha='right')
ax.set_ylabel('PPII Fraction (%)', fontsize=13, fontfamily='DejaVu Serif', fontweight='bold')
ax.set_title('PPII Conformation Fraction at Pre-glycan Residues — 3 Systems\n'
             '(orange shading = GLY hinge positions)',
             fontsize=14, fontfamily='DejaVu Serif', fontweight='bold')
ax.legend(fontsize=11, prop={'family':'DejaVu Serif','size':11})
ax.axhline(np.nanmean(ppii_data[CLR_NOG]), color=CLR_NOG, lw=1.2, ls=':', alpha=0.7)
ax.axhline(np.nanmean(ppii_data[CLR_GLY]), color=CLR_GLY, lw=1.2, ls=':', alpha=0.7)
ax.axhline(np.nanmean(ppii_data[CLR_EXT]), color=CLR_EXT, lw=1.2, ls=':', alpha=0.7)
ax.set_ylim(0, 110)
ax.tick_params(labelsize=9)
for sp in ax.spines.values(): sp.set_linewidth(1.6)

plt.tight_layout()
out = os.path.join(output_dir, 'preglycan_ppii_fraction.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

## 7 — Per-residue Cluster Membership Heatmap

In [ ]:
# For each residue, fraction in each basin — use nog and gly (have res labels)
unique_res = np.union1d(np.unique(res_nog), np.unique(res_gly))

def basin_fracs_per_res(phi_arr, psi_arr, res_arr):
    fracs = {b: [] for b in basins}
    res_list = []
    for r in sorted(np.unique(res_arr)):
        mask = res_arr == r
        if mask.sum() == 0:
            for b in basins: fracs[b].append(np.nan)
        else:
            lbs = km.predict(np.column_stack([phi_arr[mask], psi_arr[mask]]))
            for b in basins:
                frac = np.sum([CLUSTER_NAMES[c] == b for c in lbs]) / len(lbs)
                fracs[b].append(frac)
        res_list.append(r)
    return fracs, np.array(res_list)

fracs_per_nog, res_list_nog = basin_fracs_per_res(phi_nog, psi_nog, res_nog)
fracs_per_gly, res_list_gly = basin_fracs_per_res(phi_gly, psi_gly, res_gly)

fig, axes = plt.subplots(2, 2, figsize=(22, 10), dpi=120, sharex=True)
fig.set_facecolor('white')

for row_i, (sys_name, fracs_per, res_list, sys_clr) in enumerate([
    ('Non-glycosylated', fracs_per_nog, res_list_nog, CLR_NOG),
    ('Mid-level Glyc',   fracs_per_gly, res_list_gly, CLR_GLY),
]):
    for col_i, basin in enumerate(['PPII', 'β-strand']):
        ax = axes[row_i, col_i]
        vals = np.array(fracs_per[basin])
        ax.bar(res_list, vals*100, color=sys_clr, alpha=0.82,
               edgecolor='none', width=0.8)
        # mark pre-glycan positions
        for pr in PRE_RES:
            if res_list.min() <= pr <= res_list.max():
                ax.axvline(pr, color='#E74C3C', lw=0.8, alpha=0.5)
        ax.set_ylabel('Fraction (%)', fontsize=10, fontfamily='DejaVu Serif')
        ax.set_title(f'{sys_name} — {basin}',
                     fontsize=11, fontfamily='DejaVu Serif', fontweight='bold', color=sys_clr)
        ax.set_ylim(0, 105)
        ax.tick_params(labelsize=8)
        for sp in ax.spines.values(): sp.set_linewidth(1.3)

axes[-1, 0].set_xlabel('Residue Number', fontsize=11, fontfamily='DejaVu Serif', fontweight='bold')
axes[-1, 1].set_xlabel('Residue Number', fontsize=11, fontfamily='DejaVu Serif', fontweight='bold')
fig.suptitle('Per-residue PPII and β-strand Fraction — 3 Systems\n'
             '(red vertical lines = pre-glycan positions)',
             fontsize=14, fontfamily='DejaVu Serif', fontweight='bold')
plt.tight_layout()
out = os.path.join(output_dir, 'perres_basin_fraction.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', out)

## Summary

| Figure | File | What it shows |
|--------|------|---------------|
| Ramachandran KDE | `ramachandran_3sys_kde.png` | 2-D density per system |
| Ramachandran overlay | `ramachandran_overlay.png` | Contour overlay of all 3 |
| 1-D distributions | `phi_psi_distributions_1d.png` | Per-system φ/ψ histograms + KDE |
| KDE overlay | `phi_psi_kde_overlay.png` | φ and ψ KDE comparison across systems |
| Cluster scatter | `kmeans_cluster_scatter.png` | k-means clusters in φ/ψ space |
| Population bar | `cluster_population_bargraph.png` | % in each conformational basin |
| Population stacked | `cluster_population_stacked.png` | Stacked bar of all basins |
| Pre-glycan PPII | `preglycan_ppii_fraction.png` | PPII fraction at pre-glycan positions |
| Per-residue basins | `perres_basin_fraction.png` | PPII and β-strand fraction along chain |

**Key expectation**: glycosylation should increase PPII and β-strand content (backbone extension) and reduce αR-helix and turn fractions, consistent with the stiffening effect seen in curvature analysis.